In [20]:
import sys
sys.path.append('..')

In [21]:
from utils.prompts import render,list_prompts
from utils.router import pick_model
from utils.llm_client import LLMClient

In [22]:
print(list_prompts())

['zero_shot.v1', 'few_shot.v1']


In [26]:
examples = """
Message:
"We are trapped on the roof with three children. Please send help immediately."

Output:
District: None | Intent: Rescue | Priority: High


Message:
"We urgently need drinking water and food supplies in Gampaha."

Output:
District: Gampaha | Intent: Supply | Priority: High


Message:
"Breaking News: Kelani River level at 9m."

Output:
District: Colombo | Intent: Info | Priority: Low


Message:
"Thanks to everyone helping the affected families."

Output:
District: None | Intent: Other | Priority: Low
"""

In [27]:
text = "BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued."
prompt_text, spec = render(
    "few_shot.v1",
    role="crisis message classifer",
    instruction="Your task is to classify one incoming message.",
    examples=examples,
    query=f"Review: {text}",
    constraints="""
    Intent must be one of:
        - Rescue
        - Supply
        - Info
        - Other

    Priority must be:
        - High
        - Low
        """,
    format="District: [Name] | Intent: [Category] | Priority: [High/Low]"
)

model = pick_model("google", "general")
llm = LLMClient("google", model)


In [28]:
text = "BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued."
messages = [
    {
        "role": "user",
        "content": prompt_text
    }
]
response = llm.chat(messages)
print(response["text"])

District: Colombo | Intent: Info | Priority: Low
